In [ ]:
import pandas as pd
from skbio.stats.distance import mantel
from skbio import DistanceMatrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from skbio.diversity.alpha import shannon

bc_dist_path = "/Users/daniellekoby/Documents/uni/thirdYear/lab/trees_analysis/DAP_b7_BC.csv"
wuf_dist_path = "/Users/daniellekoby/Documents/uni/thirdYear/lab/trees_analysis/DAP_b7_WUF.csv"

meta_path = "/Users/daniellekoby/Documents/uni/thirdYear/lab/trees_analysis/filtered_metadata.tsv"
mito_dist_path = "/Users/daniellekoby/Documents/uni/thirdYear/lab/trees_analysis/msa_consensus_clean_headers.fa.mldist"

In [ ]:
# Mito data- preprocessing 
mito_raw = pd.read_csv(mito_dist_path, sep=r'\s+', engine='python', header=None, skiprows=[0])
meta_data = pd.read_csv(meta_path, sep=r'\t', header=0)
raw_lables_biosamples = mito_raw.iloc[:, 0].str.split('_').str[1]
raw_lables_breed = mito_raw.iloc[:, 0].str.split('_').str[2:].str.join('_')

mito_raw_values = mito_raw.iloc[:, 1:].values 

mito_df = pd.DataFrame({'biosample': raw_lables_biosamples, 'breed': raw_lables_breed})

mito_df = pd.merge(mito_df, meta_data[['biosample', 'dog']], on='biosample', how='left')


In [ ]:
# filter by breed (only purebred)
# With labrador 
# purebred_lst = np.array(["Labrador_Retriever_Purebred",
#                             "Golden_Retriever_Purebred",
#                             "German_Shepherd_Dog_Purebred",
#                             "Poodle_Purebred",
#                             "French_Bulldog_Purebred",
#                             "Australian_Shepherd_Purebred",
#                             "Great_Dane_Purebred",
#                             "Border_Collie_Purebred",
#                             "Great_Pyrenees_Purebred"])
purebred_lst = np.array(["Golden_Retriever_Purebred",
                            "German_Shepherd_Dog_Purebred",
                            "Poodle_Purebred",
                            "French_Bulldog_Purebred",
                            "Australian_Shepherd_Purebred",
                            "Great_Dane_Purebred",
                            "Border_Collie_Purebred",
                            "Great_Pyrenees_Purebred"])

top_breeds_mito_df = mito_df[mito_df['breed'].isin(purebred_lst)]

## filtering the values
mito_raw['biosample']=raw_lables_biosamples
mito_raw['breed']=raw_lables_breed
# 1. Filter the rows (labels)
top_breed_mito_raw = mito_raw[mito_raw['breed'].isin(purebred_lst)]

# 2. Get the indices of the rows that stayed
keep_indices_rows = top_breed_mito_raw.index
keep_indices_cols = [i+1 for i in keep_indices_rows]
# 3. Filter the columns of a different dataframe (or the same one) using those indices
mito_raw_filtered_breed = mito_raw.loc[keep_indices_rows, keep_indices_cols]
mito_filtered_breed_values = mito_raw_filtered_breed.values
# print(top_breeds_mito_df)


In [ ]:
 # Mito distance matrix - top dogs

# # Checking dups - found one before - same bioproject id, different srr id
# to_keep = ~mito_df['biosample'].duplicated(keep='first')
mito_filtered_dog_ids = top_breeds_mito_df['dog'].to_list()
# final_mito_values = mito_raw_values[to_keep.values][:, to_keep.values]
mito_distance_matrix = DistanceMatrix(mito_filtered_breed_values, mito_filtered_dog_ids)


In [ ]:
#  # Mito distance matrix - all dogs

# # Checking dups - found one before - same bioproject id, different srr id
# to_keep = ~mito_df['biosample'].duplicated(keep='first')
# final_mito_dog_ids = mito_df.loc[to_keep, 'dog'].to_list()
# final_mito_values = mito_raw_values[to_keep.values][:, to_keep.values]
# mito_distance_matrix = DistanceMatrix(final_mito_values, final_mito_dog_ids)


In [ ]:

# Bray curtis - preprocessing
bc_df = pd.read_csv(bc_dist_path, dtype={'dap_dog_id': str})

melted = bc_df.melt(id_vars='dap_dog_id', var_name='dap_dog_id_2', value_name='dist')
pivoted = melted.pivot(index='dap_dog_id', columns='dap_dog_id_2', values='dist')

ids = pivoted.index.astype(int).tolist()
data_array = pivoted.values

bc_distance_matrix = DistanceMatrix(data_array, ids)


In [ ]:
# check mito - bc corr

melted_bc = bc_df.melt(id_vars='dap_dog_id', var_name='dap_dog_id_2', value_name='bc_dist')

up_mito = mito_distance_matrix.to_data_frame()
up_mito.index.name = 'dap_dog_id'
melted_mito = up_mito.reset_index().melt(id_vars='dap_dog_id', var_name='dap_dog_id_2', value_name='mito_dist')

melted_bc['dap_dog_id'] = melted_bc['dap_dog_id'].astype(int)
melted_bc['dap_dog_id_2'] = melted_bc['dap_dog_id_2'].astype(int)
melted_mito['dap_dog_id'] = melted_mito['dap_dog_id'].astype(int)
melted_mito['dap_dog_id_2'] = melted_mito['dap_dog_id_2'].astype(int)


joined_dist_bc_mito = pd.merge(melted_mito, melted_bc, on=['dap_dog_id', 'dap_dog_id_2'], how='inner')
# print(joined_dist_bc_mito.head)
df_unique = joined_dist_bc_mito[joined_dist_bc_mito['dap_dog_id'] < joined_dist_bc_mito['dap_dog_id_2']].copy()

df_unique = df_unique.dropna(subset=['mito_dist', 'bc_dist'])
print(df_unique[(df_unique['dap_dog_id_2'] == 13909) & (df_unique['dap_dog_id'] == 9456)])
print(df_unique[(df_unique['dap_dog_id'] == 13909) & (df_unique['dap_dog_id_2'] == 9456)])

# print(len(joined_dist_bc_mito[(joined_dist_bc_mito['bc_dist'] == joined_dist_bc_mito['mito_dist']) & ((joined_dist_bc_mito['dap_dog_id'] == joined_dist_bc_mito['dap_dog_id_2']))]))
pearson_corr = df_unique['bc_dist'].corr(df_unique['mito_dist'], method='pearson')
spearman_corr = df_unique['bc_dist'].corr(df_unique['mito_dist'], method='spearman')

plt.scatter(x = df_unique['bc_dist'], y=df_unique['mito_dist'], alpha= 0.3)
plt.xlabel('Bray Curtis Distances')
plt.ylabel('Mitochondrial Distances')
plt.title(f'Correlation test - Mitochondria / BC : {round(spearman_corr, 4)}')
plt.show()

In [ ]:
# Run mantel 

# keep only matching dog ids
common_ids = list(set(mito_distance_matrix.ids) & set(bc_distance_matrix.ids))
print(len(common_ids))
# Run mantel (with only matching dog ids)
corr_coef, p_value, n = mantel(mito_distance_matrix.filter(common_ids), bc_distance_matrix.filter(common_ids),
                                method='spearman',
                                permutations=10000, alternative='two-sided', strict=True, lookup=None, seed=None)

print(corr_coef, p_value, n)

In [ ]:
#WUF - preprocessing
wuf_df = pd.read_csv(wuf_dist_path, dtype={'dap_dog_id': str})

melted = wuf_df.melt(id_vars='dap_dog_id', var_name='dap_dog_id_2', value_name='dist')
pivoted = melted.pivot(index='dap_dog_id', columns='dap_dog_id_2', values='dist')

ids = pivoted.index.astype(int).tolist()
data_array = pivoted.values

wuf_distance_matrix = DistanceMatrix(data_array, ids)

# filter out 
common_ids = list(set(mito_distance_matrix.ids) & set(wuf_distance_matrix.ids))
# Run mantel 
corr_coef, p_value, n = mantel(mito_distance_matrix.filter(common_ids), wuf_distance_matrix.filter(common_ids),
                method='spearman',
                permutations=9099, alternative='two-sided', strict=True, lookup=None, seed=None)

print(corr_coef, p_value, n)

In [ ]:
# check mito - wuf corr

melted_wuf = wuf_df.melt(id_vars='dap_dog_id', var_name='dap_dog_id_2', value_name='wuf_dist')

up_mito = mito_distance_matrix.to_data_frame()
up_mito.index.name = 'dap_dog_id'
melted_mito = up_mito.reset_index().melt(id_vars='dap_dog_id', var_name='dap_dog_id_2', value_name='mito_dist')

melted_wuf['dap_dog_id'] = melted_wuf['dap_dog_id'].astype(int)
melted_wuf['dap_dog_id_2'] = melted_wuf['dap_dog_id_2'].astype(int)
melted_mito['dap_dog_id'] = melted_mito['dap_dog_id'].astype(int)
melted_mito['dap_dog_id_2'] = melted_mito['dap_dog_id_2'].astype(int)


joined_dist_wuf_mito = pd.merge(melted_mito, melted_wuf, on=['dap_dog_id', 'dap_dog_id_2'], how='inner')
# print(joined_dist_bc_mito.head)
df_unique = joined_dist_wuf_mito[joined_dist_wuf_mito['dap_dog_id'] < joined_dist_wuf_mito['dap_dog_id_2']].copy()

df_unique = df_unique.dropna(subset=['mito_dist', 'wuf_dist'])

pearson_corr = df_unique['wuf_dist'].corr(df_unique['mito_dist'], method='pearson')
spearman_corr = df_unique['wuf_dist'].corr(df_unique['mito_dist'], method='spearman')

plt.scatter(x = df_unique['wuf_dist'], y=df_unique['mito_dist'], alpha= 0.3)
plt.xlabel('Weighted Unifrac Distances')
plt.ylabel('Mitochondrial Distances')
plt.title(f'Correlation Test - Mitochondria / WUF: {round(spearman_corr, 4)}')
plt.show()

In [ ]:
# --- 1. PARSE THE ITOL FILE ---
itol_path = "/Users/daniellekoby/Documents/uni/thirdYear/lab/itol_haplogroups_k9.txt"  # <--- UPDATE THIS PATH

haplo_map = []
with open(itol_path, 'r') as f:
    reading_data = False
    for line in f:
        line = line.strip()
        if line == 'DATA':
            reading_data = True
            continue
        
        if reading_data and line:
            # Format is: Long_ID, Color, Haplogroup
            parts = line.split(',')
            if len(parts) >= 3:
                long_id = parts[0]
                haplo_label = parts[2]
                
                # Extract the SAMN biosample ID (index 1 after splitting by '_')
                # Format: SRR..._SAMNxxxx_Breed...
                try:
                    biosample = long_id.split('_')[1]
                    haplo_map.append({'biosample': biosample, 'new_haplogroup': haplo_label})
                except IndexError:
                    continue

# Create a dataframe for the new groups
haplo_df = pd.DataFrame(haplo_map)

# --- 2. MERGE WITH YOUR EXISTING METADATA ---
# We merge on 'biosample' to get the 'dog' ID that matches your Distance Matrix
# meta_data is the variable from your notebook
meta_with_haplo = pd.merge(meta_data, haplo_df, on='biosample', how='inner')

# Set the index to 'dog' so it matches the Distance Matrix IDs
meta_with_haplo.set_index('dog', inplace=True)

print(f"Matched {len(meta_with_haplo)} dogs with new haplogroup assignments.")
print(meta_with_haplo['new_haplogroup'].value_counts())

In [ ]:
alpha_div_path = "/Users/daniellekoby/Documents/uni/thirdYear/lab/DAP_b1-7_ra_kraken_sp.csv"
lid_to_dog_id_path = "/Users/daniellekoby/Documents/uni/thirdYear/lab/fastq_to_dog_map.csv"
lid_to_dog_id_df = pd.read_csv(lid_to_dog_id_path)
alpha_div_lid_df = pd.read_csv(alpha_div_path)


In [ ]:
df_alpha = pd.merge(alpha_div_lid_df, lid_to_dog_id_df, on='sample_id', how='left')
# print(df_alpha.head)
# 1. Get the names of the first (0) and second-to-last (-2) columns
cols_to_drop = df_alpha.columns[[0, -2]]
# print(df_alpha[cols_to_drop])
# 2. Drop them from the DataFrame
df_alpha = df_alpha.drop(columns=cols_to_drop).dropna()

# Verify the result
print(df_alpha['dap_dog_id'].isna().sum())
df_alpha = df_alpha.set_index('dap_dog_id')
print(df_alpha.head)
df_alpha['shannon_idx'] = df_alpha.apply(shannon, axis = 1)
print(df_alpha['shannon_idx'].mean())
